In [1]:
import pandas as pd
import numpy as np
import re
import string
import warnings
import emoji
from langdetect import detect, DetectorFactory
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

warnings.filterwarnings('ignore')
DetectorFactory.seed = 0 # for consistency

## Data Cleaning

Dataset needs to be cleaned to engineer features, extract clusters, and prepare data for modeling.

In [2]:
# Read and load dataset
df = pd.read_csv('data/amazon_bestsellers_reviews.csv')
print(df.dtypes) # helpful_votes should be int, date should be datetime64
df.head()

department       object
product_index     int64
product_name     object
product_url      object
reviewer         object
rating            int64
date             object
verified         object
title            object
body             object
helpful_votes    object
variant          object
image_count       int64
dtype: object


,department,product_index,product_name,product_url,reviewer,rating,date,verified,title,body,helpful_votes,variant,image_count
0,Electronics,2,"Apple EarPods Headphones with USB-C Plug, Wire...",https://www.amazon.com/dp/B0DCH8VDXF,Sridhar Mani,5,"Reviewed in the United States on March 17, 2026",Verified Purchase,Best-in-Class Quality and Sound,These Apple EarPods with USB-C are truly best ...,28 people found this helpful,Size: One SizeStyle: USB-C,0
1,Electronics,2,"Apple EarPods Headphones with USB-C Plug, Wire...",https://www.amazon.com/dp/B0DCH8VDXF,A.S,5,"Reviewed in the United States on April 11, 2026",Verified Purchase,Great Sound and Comfort,"I bought these for my son, who prefers traditi...",8 people found this helpful,Size: One SizeStyle: USB-C,0
2,Electronics,2,"Apple EarPods Headphones with USB-C Plug, Wire...",https://www.amazon.com/dp/B0DCH8VDXF,Amazon_Customer,5,"Reviewed in the United States on April 10, 2026",Verified Purchase,"The $9 Audiophile Secret: Small Dongle, Big Pr...",Let’s be honest: nobody actually wants to live...,7 people found this helpful,Size: One SizeStyle: USB-C,0
3,Electronics,2,"Apple EarPods Headphones with USB-C Plug, Wire...",https://www.amazon.com/dp/B0DCH8VDXF,K,5,"Reviewed in the United States on March 17, 2026",Verified Purchase,better than overhead headphones,I honestly love these headphones so much. I ac...,23 people found this helpful,Size: One SizeStyle: USB-C,0
4,Electronics,2,"Apple EarPods Headphones with USB-C Plug, Wire...",https://www.amazon.com/dp/B0DCH8VDXF,Low,5,"Reviewed in the United States on April 25, 2026",Verified Purchase,Worth the extra money. They're the best ones I...,"Best you can buy. I have 4 usbc ear buds, thes...",NaN,Size: One SizeStyle: USB-C,0


In [3]:
# Split date into date and location, parse date
pattern = r'Reviewed in (.+?) on (.+)'
df[['location', 'date']] = df['date'].str.extract(pattern)
df['location'] = df['location'].str.replace(r'(?i)^the\s+', '', regex=True)
df['date'] = pd.to_datetime(df['date'])
df.head()

,department,product_index,product_name,product_url,reviewer,rating,date,verified,title,body,helpful_votes,variant,image_count,location
0,Electronics,2,"Apple EarPods Headphones with USB-C Plug, Wire...",https://www.amazon.com/dp/B0DCH8VDXF,Sridhar Mani,5,2026-03-17,Verified Purchase,Best-in-Class Quality and Sound,These Apple EarPods with USB-C are truly best ...,28 people found this helpful,Size: One SizeStyle: USB-C,0,United States
1,Electronics,2,"Apple EarPods Headphones with USB-C Plug, Wire...",https://www.amazon.com/dp/B0DCH8VDXF,A.S,5,2026-04-11,Verified Purchase,Great Sound and Comfort,"I bought these for my son, who prefers traditi...",8 people found this helpful,Size: One SizeStyle: USB-C,0,United States
2,Electronics,2,"Apple EarPods Headphones with USB-C Plug, Wire...",https://www.amazon.com/dp/B0DCH8VDXF,Amazon_Customer,5,2026-04-10,Verified Purchase,"The $9 Audiophile Secret: Small Dongle, Big Pr...",Let’s be honest: nobody actually wants to live...,7 people found this helpful,Size: One SizeStyle: USB-C,0,United States
3,Electronics,2,"Apple EarPods Headphones with USB-C Plug, Wire...",https://www.amazon.com/dp/B0DCH8VDXF,K,5,2026-03-17,Verified Purchase,better than overhead headphones,I honestly love these headphones so much. I ac...,23 people found this helpful,Size: One SizeStyle: USB-C,0,United States
4,Electronics,2,"Apple EarPods Headphones with USB-C Plug, Wire...",https://www.amazon.com/dp/B0DCH8VDXF,Low,5,2026-04-25,Verified Purchase,Worth the extra money. They're the best ones I...,"Best you can buy. I have 4 usbc ear buds, thes...",NaN,Size: One SizeStyle: USB-C,0,United States


In [4]:
# Clean helpful_votes and create target column
pattern = r'(One|\d+)'
df['helpful_votes'] = df['helpful_votes'].str.extract(pattern)[0]
df['helpful_votes'] = df['helpful_votes'].replace('One', 1)
df['helpful_votes'] = pd.to_numeric(df['helpful_votes'], errors='coerce').fillna(0).astype(int)
df['is_helpful'] = (df['helpful_votes'] > 0).astype(int) # Assign 1 or 0

print(df.shape)
display(df.head())

(1784, 15)


,department,product_index,product_name,product_url,reviewer,rating,date,verified,title,body,helpful_votes,variant,image_count,location,is_helpful
0,Electronics,2,"Apple EarPods Headphones with USB-C Plug, Wire...",https://www.amazon.com/dp/B0DCH8VDXF,Sridhar Mani,5,2026-03-17,Verified Purchase,Best-in-Class Quality and Sound,These Apple EarPods with USB-C are truly best ...,28,Size: One SizeStyle: USB-C,0,United States,1
1,Electronics,2,"Apple EarPods Headphones with USB-C Plug, Wire...",https://www.amazon.com/dp/B0DCH8VDXF,A.S,5,2026-04-11,Verified Purchase,Great Sound and Comfort,"I bought these for my son, who prefers traditi...",8,Size: One SizeStyle: USB-C,0,United States,1
2,Electronics,2,"Apple EarPods Headphones with USB-C Plug, Wire...",https://www.amazon.com/dp/B0DCH8VDXF,Amazon_Customer,5,2026-04-10,Verified Purchase,"The $9 Audiophile Secret: Small Dongle, Big Pr...",Let’s be honest: nobody actually wants to live...,7,Size: One SizeStyle: USB-C,0,United States,1
3,Electronics,2,"Apple EarPods Headphones with USB-C Plug, Wire...",https://www.amazon.com/dp/B0DCH8VDXF,K,5,2026-03-17,Verified Purchase,better than overhead headphones,I honestly love these headphones so much. I ac...,23,Size: One SizeStyle: USB-C,0,United States,1
4,Electronics,2,"Apple EarPods Headphones with USB-C Plug, Wire...",https://www.amazon.com/dp/B0DCH8VDXF,Low,5,2026-04-25,Verified Purchase,Worth the extra money. They're the best ones I...,"Best you can buy. I have 4 usbc ear buds, thes...",0,Size: One SizeStyle: USB-C,0,United States,0


In [5]:
# Text Preprocessing (Title + Body)

# Filter non-english reviews
def is_english(text):
    try:
        return detect(str(text)) == 'en'
    except:
        return False

df = df[df['body'].apply(is_english)].copy()
df = df.reset_index(drop=True)

# Decode emojis and remove non-ascii characters
df['title'] = df['title'].apply(lambda x: emoji.demojize(str(x)))
df['body'] = df['body'].apply(lambda x: emoji.demojize(str(x)))
df['title'] = df['title'].str.encode('ascii', 'ignore').str.decode('ascii')
df['body'] = df['body'].str.encode('ascii', 'ignore').str.decode('ascii')

# Preparing Text for LSA
df['text_for_lsa'] = df['title'].fillna('') + ' ' + df['body'].fillna('')
df['text_for_lsa'] = df['text_for_lsa'].str.lower()
df['text_for_lsa'] = df['text_for_lsa'].str.replace(f"[{re.escape(string.punctuation)}]", " ", regex=True)
df['text_for_lsa'] = df['text_for_lsa'].str.replace(r'\d+', '', regex=True)

# Removing Stopwords
def remove_stopwords(text):
    words = [word for word in text.split() if word not in ENGLISH_STOP_WORDS]
    return ' '.join(words)

df['text_for_lsa'] = df['text_for_lsa'].apply(remove_stopwords)
df['text_for_lsa'] = df['text_for_lsa'].str.strip().str.replace(r'\s+', ' ', regex=True)

print(df.shape)
display(df.head())

(1513, 16)


,department,product_index,product_name,product_url,reviewer,rating,date,verified,title,body,helpful_votes,variant,image_count,location,is_helpful,text_for_lsa
0,Electronics,2,"Apple EarPods Headphones with USB-C Plug, Wire...",https://www.amazon.com/dp/B0DCH8VDXF,Sridhar Mani,5,2026-03-17,Verified Purchase,Best-in-Class Quality and Sound,These Apple EarPods with USB-C are truly best ...,28,Size: One SizeStyle: USB-C,0,United States,1,best class quality sound apple earpods usb c t...
1,Electronics,2,"Apple EarPods Headphones with USB-C Plug, Wire...",https://www.amazon.com/dp/B0DCH8VDXF,A.S,5,2026-04-11,Verified Purchase,Great Sound and Comfort,"I bought these for my son, who prefers traditi...",8,Size: One SizeStyle: USB-C,0,United States,1,great sound comfort bought son prefers traditi...
2,Electronics,2,"Apple EarPods Headphones with USB-C Plug, Wire...",https://www.amazon.com/dp/B0DCH8VDXF,Amazon_Customer,5,2026-04-10,Verified Purchase,"The $9 Audiophile Secret: Small Dongle, Big Pr...",Lets be honest: nobody actually wants to live ...,7,Size: One SizeStyle: USB-C,0,United States,1,audiophile secret small dongle big practicalit...
3,Electronics,2,"Apple EarPods Headphones with USB-C Plug, Wire...",https://www.amazon.com/dp/B0DCH8VDXF,K,5,2026-03-17,Verified Purchase,better than overhead headphones,I honestly love these headphones so much. I ac...,23,Size: One SizeStyle: USB-C,0,United States,1,better overhead headphones honestly love headp...
4,Electronics,2,"Apple EarPods Headphones with USB-C Plug, Wire...",https://www.amazon.com/dp/B0DCH8VDXF,Low,5,2026-04-25,Verified Purchase,Worth the extra money. They're the best ones I...,"Best you can buy. I have 4 usbc ear buds, thes...",0,Size: One SizeStyle: USB-C,0,United States,0,worth extra money best ones best buy usbc ear ...


## Feature Engineering: Encoding Categorical Variables

Before SVD and clustering, all categorical columns are converted to numerical form to ensure
ML compatibility. `helpful_votes` is deliberately excluded from the feature set here (kept
in the DataFrame but **not used as a model feature**) to prevent data leakage into `is_helpful`.

In [6]:
# ── Encode categorical features for ML ──────────────────────────────────────
# 'helpful_votes' is excluded from the feature table to avoid data leakage
# (is_helpful was derived from it), but it REMAINS in df for reference.

# Rating: already numeric (1–5); cast to int for safety
df['rating'] = pd.to_numeric(df['rating'], errors='coerce').fillna(df['rating'].median()).astype(int)

# Date-derived features (year, month, day-of-week)
df['year']        = df['date'].dt.year.astype(int)
df['month']       = df['date'].dt.month.astype(int)
df['day_of_week'] = df['date'].dt.dayofweek.astype(int)

# Review Age Days (days since reviews were scraped)
scrape_date = pd.Timestamp('2026-05-02')
df['review_age_days'] = (scrape_date - df['date']).dt.days

# Verified purchase: boolean → int (1/0)
if 'verified_purchase' in df.columns:
    df['verified_purchase'] = df['verified_purchase'].map({True: 1, False: 0, 'True': 1, 'False': 0})
    df['verified_purchase'] = pd.to_numeric(df['verified_purchase'], errors='coerce').fillna(0).astype(int)

# Title / Body length features (text remains; lengths are numeric)
df['title_len'] = df['title'].fillna('').str.len().astype(int)
df['body_len']  = df['body'].fillna('').str.len().astype(int)
df['word_count'] = df['text_for_lsa'].str.split().str.len().fillna(0).astype(int)
df['avg_word_len'] = df['body_len'] / df['word_count']

# Sentiment Analysis
analyzer = SentimentIntensityAnalyzer()
df['sentiment'] = df['body'].fillna('').apply(
    lambda x: analyzer.polarity_scores(x)['compound'])
df['sentiment_extremity'] = df['sentiment'].abs()

print("DataFrame dtypes after encoding:")
print(df.dtypes)
print(f"\nDataFrame shape: {df.shape}")

DataFrame dtypes after encoding:
department                     object
product_index                   int64
product_name                   object
product_url                    object
reviewer                       object
rating                          int64
date                   datetime64[ns]
verified                       object
title                          object
body                           object
helpful_votes                   int64
variant                        object
image_count                     int64
location                       object
is_helpful                      int64
text_for_lsa                   object
year                            int64
month                           int64
day_of_week                     int64
review_age_days                 int64
title_len                       int64
body_len                        int64
word_count                      int64
avg_word_len                  float64
sentiment                     float64
sentiment_extremi

## Export Dataset for TF-IDF, LSA and Clustering

In [7]:
print(f"Final dataset shape: {df.shape}")
print(f"\nAll dtypes numerical? {all(df.dtypes.apply(lambda x: np.issubdtype(x, np.number)))}")
print(f"\nColumn list ({len(df.columns)} columns):")
print(list(df.columns))
print("\nSample:")
df.head()

Final dataset shape: (1513, 26)

All dtypes numerical? False

Column list (26 columns):
['department', 'product_index', 'product_name', 'product_url', 'reviewer', 'rating', 'date', 'verified', 'title', 'body', 'helpful_votes', 'variant', 'image_count', 'location', 'is_helpful', 'text_for_lsa', 'year', 'month', 'day_of_week', 'review_age_days', 'title_len', 'body_len', 'word_count', 'avg_word_len', 'sentiment', 'sentiment_extremity']

Sample:


,department,product_index,product_name,product_url,reviewer,rating,date,verified,title,body,...,year,month,day_of_week,review_age_days,title_len,body_len,word_count,avg_word_len,sentiment,sentiment_extremity
0,Electronics,2,"Apple EarPods Headphones with USB-C Plug, Wire...",https://www.amazon.com/dp/B0DCH8VDXF,Sridhar Mani,5,2026-03-17,Verified Purchase,Best-in-Class Quality and Sound,These Apple EarPods with USB-C are truly best ...,...,2026,3,1,46,31,557,61,9.131148,0.9876,0.9876
1,Electronics,2,"Apple EarPods Headphones with USB-C Plug, Wire...",https://www.amazon.com/dp/B0DCH8VDXF,A.S,5,2026-04-11,Verified Purchase,Great Sound and Comfort,"I bought these for my son, who prefers traditi...",...,2026,4,5,21,23,495,43,11.511628,0.9547,0.9547
2,Electronics,2,"Apple EarPods Headphones with USB-C Plug, Wire...",https://www.amazon.com/dp/B0DCH8VDXF,Amazon_Customer,5,2026-04-10,Verified Purchase,"The $9 Audiophile Secret: Small Dongle, Big Pr...",Lets be honest: nobody actually wants to live ...,...,2026,4,4,22,56,1369,135,10.140741,0.9857,0.9857
3,Electronics,2,"Apple EarPods Headphones with USB-C Plug, Wire...",https://www.amazon.com/dp/B0DCH8VDXF,K,5,2026-03-17,Verified Purchase,better than overhead headphones,I honestly love these headphones so much. I ac...,...,2026,3,1,46,31,935,82,11.402439,0.9953,0.9953
4,Electronics,2,"Apple EarPods Headphones with USB-C Plug, Wire...",https://www.amazon.com/dp/B0DCH8VDXF,Low,5,2026-04-25,Verified Purchase,Worth the extra money. They're the best ones I...,"Best you can buy. I have 4 usbc ear buds, thes...",...,2026,4,5,7,51,132,20,6.600000,0.9705,0.9705


In [8]:
# ── Sanity Checks ───────────────────────────────────────────────────────────

# Catch categorical columns
cat_cols = df.select_dtypes(include=['object', 'category']).columns.tolist()
print(f"Remaining categorical columns: {cat_cols if cat_cols else 'None — all numerical ✓'}")

# Catch missing values
print("\nMissing values per column (top offenders):")
missing = df.isnull().sum()
print(missing[missing > 0] if missing.any() else "  None ✓")

# Catch zero variance columns
zero_var = [col for col in df.columns if df[col].nunique() <= 1]
print("\nZero variance columns:", zero_var if zero_var else "None ✓")

# Class Balance Check
print("\nTarget Distribution:")
print(df['is_helpful'].value_counts(normalize=True).round(3), '\n')

# Fill any edge-case NaNs and infs
df_final = df.fillna(0)
df_final.replace([np.inf, -np.inf], 0, inplace=True)

# Check for remaining NaN or inf values
has_nan = df_final.isna().any().any()
has_inf = np.isinf(df_final.select_dtypes(include=[np.number])).any().any()

if has_nan or has_inf:
    print("Warning: Data still contains NaN or inf values!")
    print(f"NaN present: {has_nan}, Inf present: {has_inf}")
else:
    print("No NaN or inf values detected.")

print("\nFinal shape after NaN and inf fill:", df.shape, '\n')

Remaining categorical columns: ['department', 'product_name', 'product_url', 'reviewer', 'verified', 'title', 'body', 'variant', 'location', 'text_for_lsa']

Missing values per column (top offenders):
variant    70
dtype: int64

Zero variance columns: ['verified']

Target Distribution:
is_helpful
1    0.611
0    0.389
Name: proportion, dtype: float64 

No NaN or inf values detected.

Final shape after NaN and inf fill: (1513, 26) 



In [9]:
# ── Export final dataset ─────────────────────────────────────────────────────
df.to_csv("data/amazon_bestsellers_reviews_cleaned.csv", index=False)

print("Exported ✓")

Exported ✓
